# Solution 3 — Dedicated Arabic encoders

DarijaBERT's win was an artifact: it collapses to R@5 **0.645** on real Wikipedia text because it was pretrained on Darija social media, not formal MSA. And no mitigation so far beats raw Darija significantly.

This tests purpose-built **Arabic** embedding models — including **GATE-AraBert-v1**, named in your proposal but never evaluated. All are sentence-transformers models with properly trained pooling, unlike the manual mean-pooling that likely handicapped DarijaBERT.

**Primary evaluation is the Wikipedia subset** (200 items, no authorship confound). Pilot is reported alongside for comparison.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **No API key.** GPU runtime recommended.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers requests

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "repo_raw": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main",
    "encoders": [
        # Arabic-specific sentence-transformers models
        ("Omartificial-Intelligence-Space/GATE-AraBert-v1",              ""),
        ("Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2", ""),
        ("Omartificial-Intelligence-Space/Arabert-all-nli-triplet-Matryoshka", ""),
        ("Omartificial-Intelligence-Space/AraGemma-Embedding-300m",      ""),
        # Best general model from the previous run, kept as the control
        ("intfloat/multilingual-e5-base",                                "e5"),
    ],
    "alphas": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    "k_values": (1, 3, 5, 10),
    "max_k": 10,
    "bootstrap_n": 1000,
    "ci": 95,
}

### Load corpus and both question sets

In [ ]:
import json, requests

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)
pilot_qa = json.loads(requests.get(f"{CONFIG['repo_raw']}/data/qa_pairs.json").text)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
known = set(corpus_ids)

SUBSETS = {
    "wikipedia": [q for q in wiki_qa if q["source_chunk_id"] in known],   # primary
    "pilot":     [q for q in pilot_qa if q["source_chunk_id"] in known],  # comparison
}
print(f"Corpus: {len(corpus)} passages")
for k, v in SUBSETS.items():
    print(f"  {k:<10} {len(v)} questions")

### Arabic normalization + BM25 + rule-based mitigation

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])

DARIJA_TO_MSA = {
    "شنو هي": "ما هي", "شنو هو": "ما هو", "شنو هما": "ما هي", "شنو هوما": "ما هي",
    "شحال ديال": "كم", "بشحال": "بكم", "شحال": "كم",
    "فين": "أين", "لفين": "إلى أين", "منين": "من أين",
    "علاش": "لماذا", "علاه": "لماذا", "كيفاش": "كيف",
    "إمتى": "متى", "فوقاش": "متى", "شكون": "من", "أشمن": "أي",
    "فأش": "في أي", "فأي": "في أي", "بأش": "بماذا", "أش": "ماذا", "شنو": "ما",
    "واش": "هل", "كاينة": "توجد", "كاين": "يوجد", "ماكاينش": "لا يوجد",
    "بزاف": "كثيرا", "دابا": "الآن", "غادي": "سوف", "باش": "لكي",
    "هادشي": "هذا", "هادي": "هذه", "هاد": "هذا",
    "بحال": "مثل", "حيت": "لأن", "ملي": "عندما", "واخا": "رغم",
    "ماشي": "ليس", "بلا": "بدون", "ديالو": "", "ديالها": "", "ديالهم": "", "ديال": "",
}

def rule_normalize(text):
    out = text
    for d, m in sorted(DARIJA_TO_MSA.items(), key=lambda kv: -len(kv[0])):
        out = re.sub(rf"(?<!\w){re.escape(d)}(?!\w)", m, out)
    return re.sub(r"\s+", " ", out).strip()

for items in SUBSETS.values():
    for it in items:
        it["M4_rulebased"] = rule_normalize(it["darija_query"])
        it["M3_rule_expansion"] = f'{it["darija_query"]} {it["M4_rulebased"]}'

VARIANTS = ["msa_query", "darija_query", "M4_rulebased", "M3_rule_expansion"]
print("BM25 index and mitigation variants ready.")

### Encoding + retrieval

In [ ]:
import torch, gc
from sentence_transformers import SentenceTransformer

def load_encoder(model_id, style):
    model = SentenceTransformer(model_id)
    pp, pq = ("passage: ", "query: ") if style == "e5" else ("", "")
    emb = model.encode([pp + t for t in corpus_texts],
                       normalize_embeddings=True, show_progress_bar=True, batch_size=32)
    return (lambda q: model.encode([pq + q], normalize_embeddings=True)[0]), \
           np.asarray(emb, "float32"), model

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def make_retriever(encode_query, corpus_emb):
    def retrieve(q, k, alpha):
        s = 0.0
        if alpha > 0:
            s = alpha * minmax(corpus_emb @ encode_query(q))
        if alpha < 1:
            s = s + (1 - alpha) * minmax(np.asarray(bm25.get_scores(tokenize(q))))
        return [corpus_ids[i] for i in np.argsort(-s)[:k]]
    return retrieve

def per_item(retrieve, items, field, alpha):
    hits = {k: [] for k in CONFIG["k_values"]}
    rr = []
    for it in items:
        got = retrieve(it[field], CONFIG["max_k"], alpha)
        gold = it["source_chunk_id"]
        for k in CONFIG["k_values"]:
            hits[k].append(1.0 if gold in got[:k] else 0.0)
        rr.append(1.0 / (got.index(gold) + 1) if gold in got else 0.0)
    return {**{f"R@{k}": np.array(v) for k, v in hits.items()}, "MRR": np.array(rr)}

### Run the grid

In [ ]:
import pandas as pd

raw = {}      # (encoder, subset, variant, alpha) -> per-item arrays
sweep = []    # tidy rows for the alpha sweep

for model_id, style in CONFIG["encoders"]:
    short = model_id.split("/")[-1]
    print(f"\n=== {short} ===")
    try:
        encode_query, corpus_emb, model = load_encoder(model_id, style)
    except Exception as e:
        print(f"  SKIPPED ({type(e).__name__}: {e})")
        continue

    retrieve = make_retriever(encode_query, corpus_emb)
    for alpha in CONFIG["alphas"]:
        for subset, items in SUBSETS.items():
            for v in VARIANTS:
                arr = per_item(retrieve, items, v, alpha)
                raw[(short, subset, v, alpha)] = arr
                sweep.append({"encoder": short, "subset": subset, "variant": v,
                              "alpha": alpha,
                              **{m: a.mean() for m, a in arr.items()}})
        print(f"  alpha={alpha} done")

    del encode_query, corpus_emb, model, retrieve
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sweep_df = pd.DataFrame(sweep)
sweep_df.to_csv("arabic_encoder_sweep.csv", index=False)
print(f"\n{len(sweep_df)} rows -> arabic_encoder_sweep.csv")

### Alpha behaviour on the primary (wikipedia) subset

In [ ]:
wiki_dar = sweep_df[(sweep_df.subset == "wikipedia") & (sweep_df.variant == "darija_query")]
print("Darija R@5 vs alpha — WIKIPEDIA subset (primary)\n")
print(wiki_dar.pivot_table(index="alpha", columns="encoder", values="R@5").round(3).to_string())
print("\nA column that RISES with alpha means that encoder's dense signal helps.")

### Best alpha per encoder, and the resulting dialect gap

In [ ]:
best_alpha = {}
print("\nBest alpha per encoder (by wikipedia Darija R@5):\n")
for enc in wiki_dar.encoder.unique():
    sub = wiki_dar[wiki_dar.encoder == enc]
    a = float(sub.loc[sub["R@5"].idxmax(), "alpha"])
    best_alpha[enc] = a
    row = sub[sub.alpha == a].iloc[0]
    print(f"  {enc:<42} alpha={a}  R@1={row['R@1']:.3f}  R@5={row['R@5']:.3f}  MRR={row['MRR']:.3f}")

### Bootstrap CIs on the dialect gap and on mitigation

In [ ]:
rng = np.random.default_rng(42)

def boot_diff(a, b):
    d = a - b
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    means = d[idx].mean(axis=1)
    lo, hi = np.percentile(means, [(100 - CONFIG["ci"]) / 2, 100 - (100 - CONFIG["ci"]) / 2])
    return d.mean(), lo, hi

gap_rows, mit_rows = [], []
for enc, a in best_alpha.items():
    for subset in SUBSETS:
        km = (enc, subset, "msa_query", a)
        kd = (enc, subset, "darija_query", a)
        if km not in raw:
            continue
        for metric in ["R@1", "R@5", "MRR"]:
            g, lo, hi = boot_diff(raw[km][metric], raw[kd][metric])
            gap_rows.append({"encoder": enc, "subset": subset, "alpha": a, "metric": metric,
                             "gap": g, "lo": lo, "hi": hi,
                             "significant": "yes" if lo > 0 else "no"})
        drop = raw[km]["R@5"].mean() - raw[kd]["R@5"].mean()
        for v in ["M4_rulebased", "M3_rule_expansion"]:
            d, lo, hi = boot_diff(raw[(enc, subset, v, a)]["R@5"], raw[kd]["R@5"])
            mit_rows.append({"encoder": enc, "subset": subset, "mitigation": v,
                             "improvement": d, "lo": lo, "hi": hi,
                             "recovery_%": (d / drop * 100) if drop > 0 else float("nan"),
                             "significant": "yes" if lo > 0 else "no"})

gaps = pd.DataFrame(gap_rows)
mits = pd.DataFrame(mit_rows)
gaps.to_csv("arabic_gaps.csv", index=False)
mits.to_csv("arabic_mitigation.csv", index=False)

print("=== DIALECT GAP, wikipedia subset (95% CI) ===\n")
print(gaps[gaps.subset == "wikipedia"].sort_values(["metric", "gap"])
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

### Headline: which encoder minimises the dialect gap?

In [ ]:
print("\n=== RANKING: smallest dialect gap on the wikipedia subset (R@5) ===\n")
r5 = gaps[(gaps.subset == "wikipedia") & (gaps.metric == "R@5")].copy()
for enc in r5.encoder:
    a = best_alpha[enc]
    r5.loc[r5.encoder == enc, "msa_R@5"] = raw[(enc, "wikipedia", "msa_query", a)]["R@5"].mean()
    r5.loc[r5.encoder == enc, "darija_R@5"] = raw[(enc, "wikipedia", "darija_query", a)]["R@5"].mean()

print(r5[["encoder", "alpha", "msa_R@5", "darija_R@5", "gap", "lo", "hi", "significant"]]
      .sort_values("gap").to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nA smaller gap means the encoder represents Darija and MSA more similarly.")
print("A gap whose CI includes zero would mean dialect no longer measurably hurts")
print("retrieval with that encoder — the strongest possible result here.")

### Mitigation check (does anything beat raw Darija significantly?)

In [ ]:
print("\n=== MITIGATION vs raw Darija (95% CI) ===\n")
print(mits.sort_values("recovery_%", ascending=False)
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nAny row with significant = yes is the first real mitigation result.")

from google.colab import files
files.download("arabic_encoder_sweep.csv")
files.download("arabic_gaps.csv")
files.download("arabic_mitigation.csv")